In [ ]:
import os
import shutil
from PIL import Image

def get_unique_save_path(directory, filename):
    name, ext = os.path.splitext(filename)
    save_path = os.path.join(directory, filename)
    counter = 1
    while os.path.exists(save_path):
        save_path = os.path.join(directory, f"{name}_{counter}{ext}")
        counter += 1
    return save_path

def step1_extract_and_convert(source_root, intermediate_dir):
    print(f"\n>>> [Stage 1] Started: Traversing source folder and extracting -> {intermediate_dir}")
    
    if not os.path.exists(intermediate_dir):
        os.makedirs(intermediate_dir)
        
    valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp', '.gif'}
    count_copy = 0
    count_convert = 0
    
    for root, dirs, files in os.walk(source_root):
        for filename in files:
            ext = os.path.splitext(filename)[1].lower()
            if ext in valid_exts:
                src_path = os.path.join(root, filename)
                final_name_no_ext = os.path.splitext(filename)[0]
                final_filename = final_name_no_ext + ".jpg"
                dst_path = get_unique_save_path(intermediate_dir, final_filename)
                
                try:
                    if ext == '.jpg':
                        shutil.copy(src_path, dst_path)
                        count_copy += 1
                    else:
                        with Image.open(src_path) as img:
                            img = img.convert('RGB')
                            img.save(dst_path, format='JPEG', quality=95)
                        count_convert += 1
                    
                    total_processed = count_copy + count_convert
                    if total_processed % 100 == 0:
                        print(f"  Extracted {total_processed} images (Direct Copy: {count_copy}, Converted: {count_convert})...")
                            
                except Exception as e:
                    print(f"  [Extraction Error] {src_path}: {e}")
        
    print(f">>> [Stage 1] Completed.")
    print(f"  Direct Copy (.jpg): {count_copy}")
    print(f"  Format Conversion: {count_convert}")

if __name__ == "__main__":
    SOURCE_ROOT = 'dir_source'
    INTERMEDIATE_DIR = 'dir_intermediate'
    
    # NOTE: The unscaled images generated in Stage 1 will also be utilized for objective event description 
    # generation via Large Language Models (LLMs). Preserving the original resolution ensures that the LLMs 
    # can capture fine-grained visual features and contextual details, which might otherwise be blurred or 
    # lost if the images were downscaled prematurely.
    step1_extract_and_convert(SOURCE_ROOT, INTERMEDIATE_DIR)

In [ ]:
import os
import cv2
import numpy as np
import glob

def preprocess_image(image_path, target_size, fill_color):
    img = cv2.imread(image_path)
    if img is None:
        try:
            img = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8), cv2.IMREAD_COLOR)
        except Exception:
            pass
            
    if img is None:
        raise FileNotFoundError(f"Image {image_path} not found or unreadable")

    original_height, original_width = img.shape[:2]
    scale_ratio = min(target_size[0] / original_width, target_size[1] / original_height)
    new_width = int(original_width * scale_ratio)
    new_height = int(original_height * scale_ratio)

    resized_img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_LINEAR)
    filled_img = np.full((target_size[1], target_size[0], 3), fill_color, dtype=np.uint8)
    filled_img[0:new_height, 0:new_width] = resized_img
    
    return filled_img

def process_image_folder(input_folder, output_folder, target_size=(224, 224), fill_color=(128, 128, 128), extensions=["*.jpg"]):
    print(f"\n>>> [Stage 2] Started: Reading images and processing -> {output_folder}")

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    all_image_paths = []
    for ext in extensions:
        all_image_paths.extend(glob.glob(os.path.join(input_folder, ext)))
        all_image_paths.extend(glob.glob(os.path.join(input_folder, ext.upper())))
        
    if not all_image_paths:
        print(f"No images found in {input_folder}")
        return
    
    total_images = len(all_image_paths)
    print(f"Found {total_images} images to process")
    
    for i, image_path in enumerate(all_image_paths):
        if i % 500 == 0:
            print(f"  Processing index: {i} ...")
        try:
            filename = os.path.basename(image_path)
            output_path = os.path.join(output_folder, filename)
            
            processed_img = preprocess_image(image_path, target_size, fill_color)
            is_success, buffer = cv2.imencode(".jpg", processed_img)
        
            if is_success:
                buffer.tofile(output_path)
            else:
                print(f"Encoding failed for: {filename}")
            
            if (i + 1) % 100 == 0 or i + 1 == total_images:
                print(f"Processed {i+1}/{total_images} images")
                
        except Exception as e:
            print(f"Error processing image {image_path}: {str(e)}")
    
    print(f"Processing completed. All results saved to: {output_folder}")

if __name__ == "__main__":
    INTERMEDIATE_DIR = 'dir_intermediate'
    FINAL_DIR = 'result_dir'
    
    process_image_folder(
        input_folder=INTERMEDIATE_DIR, 
        output_folder=FINAL_DIR,
        target_size=(224, 224),   
        fill_color=(128, 128, 128),   
        extensions=["*.jpg"]          
    )